# Module 5 project-exercise demo — Attention memory and the cost of selecting positions

**Implementation demo, not a completed full-scale project.** Two experiments: analytical array accounting and two-stage retrieval on synthetic data. The selector illustrates a principle; it is not a trained DSA indexer.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'


## Demo 1. Account for the arrays

Compare selected array sizes as sequence length grows. The direct path retains the output plus full score and probability matrices. The streaming estimate retains the output and the listed buffers for one tile. Neither expression includes Q/K/V, the backward pass, or runtime overhead. Identify the quadratic term and predict which curve grows faster before plotting it.

In [ ]:
lengths = np.array([1024, 2048, 4096, 8192, 16384])
heads, width, tile, bytes_per = 16, 64, 128, 4
output = heads * lengths * width * bytes_per
dense = output + 2 * heads * lengths**2 * bytes_per
stream = output + heads * (tile**2 + tile * width + 2 * tile) * bytes_per
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(figsize=(11, 5))
ax.loglog(lengths, dense / 2**20, "o-", color="#5E81AC", label="output + full scores/probabilities")
ax.loglog(lengths, stream / 2**20, "o-", color="#A3BE8C", label="output + selected tile buffers")
ax.set(xlabel="sequence length", ylabel="analytical MiB", title="Selected arrays, not measured GPU peak memory")
ax.legend(); ax.grid(alpha=.25); plt.show()
assert np.all(dense > stream)
print("Excludes Q/K/V, allocator overhead, backward, and external masks. Tile buffers depend on kernel scheduling.")

At fixed tile size, tile buffers stay constant while the output grows linearly. The two square matrices dominate the direct estimate. This is array accounting, not a measured GPU peak: scheduling and additional buffers can change the complete working set.

## Demo 2. Count the cost of a shortlist

Construct each query by adding noise to a known key. Use just eight coordinates to select 64 candidates, then score those candidates in the full 32-dimensional space. Measure shortlist recall, final retrieval, and the difference from the full attention output. These outcomes answer different questions: retrieving the best key does not necessarily preserve the complete weighted mixture.

In [ ]:
rng = np.random.default_rng(SEED)
trials, length, dim, index_rank, topk = 200, 2048, 32, 8, 64
recall = hits = full_hits = 0
relative_errors, retained_mass = [], []
def softmax_row(x):
    e = np.exp(x - x.max())
    return e / e.sum()
for _ in range(trials):
    keys = rng.normal(size=(length, dim))
    values = rng.normal(size=(length, 6))
    target = rng.integers(length)
    query = keys[target] + 0.05 * rng.normal(size=dim)
    index_scores = keys[:, :index_rank] @ query[:index_rank]
    shortlist = np.argpartition(index_scores, -topk)[-topk:]
    scores = keys @ query / np.sqrt(dim)
    recall += int(target in shortlist)
    hits += int(shortlist[np.argmax(scores[shortlist])] == target)
    full_hits += int(np.argmax(scores) == target)
    p_full, p_short = softmax_row(scores), softmax_row(scores[shortlist])
    out_full, out_short = p_full @ values, p_short @ values[shortlist]
    retained_mass.append(p_full[shortlist].sum())
    relative_errors.append(np.linalg.norm(out_short-out_full) / max(np.linalg.norm(out_full), 1e-12))
score_ratio = (length * index_rank + topk * dim) / (length * dim)
assert score_ratio == .28125
assert 0 <= hits <= recall <= trials
print(f"shortlist recall={recall/trials:.3f}; final retrieval={hits/trials:.3f}; full retrieval={full_hits/trials:.3f}")
print(f"mean retained probability={np.mean(retained_mass):.3f}; mean relative output error={np.mean(relative_errors):.3f}")
print(f"score multiplications ratio={score_ratio:.5f}; final scoring alone={topk/length:.5f}")
print("The ratio excludes top-k, learned projections, softmax, value aggregation and data movement; it is not a speedup.")

Read the metrics together. The two scoring stages use 8/32 + 64/2048 = 0.28125 of the full dot-product count; counting selected positions alone would give only 0.03125. Top-k, projections, value aggregation, and data movement are still excluded. Reduce the candidate count and measure how shortlist recall and output error change rather than assuming a particular result on this random sample.

## Scaling to the full project

For GPU measurements, match inputs and masks, warm up, synchronize and reset peak counters. State whether projections, selection, main attention and cache are included. Retrieval accuracy and attention-output error are different metrics.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.